# Anomalyy: Stock Market Anomaly Detection
### CS 210: Data Management for Data Science
**Author:** Purabh Singh

This notebook reproduces every SQL query and every visualization referenced in `final_report.md`. It reads from the persisted SQLite database (`anomalyy.db`) populated by `main.py`, so a grader can run it end-to-end without GCP/BigQuery credentials.

**Tickers analyzed:** AAPL, MSFT, TSLA, AMZN, ^GSPC (S&P 500 index)  
**Window:** 2015-01-01 to 2024-12-31  
**Detection:** Z-Score + Isolation Forest + LOF, agreement ≥ 2  
**Classification:** macroeconomic_event → sector_contagion → vader_sentiment_spike → unexplained

In [ ]:
import sys, sqlite3, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DB_PATH = '../anomalyy.db'
conn = sqlite3.connect(DB_PATH)
tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")]
print('Connected to', DB_PATH)
print('Tables:', tables)

## 1. Basic SQL Aggregate Queries (6)

Six queries that exercise basic `SELECT`, `GROUP BY`, and aggregate functionality across the four normalized tables. Each cell prints its result so the grader can verify.

In [ ]:
# Q1: Trading days per ticker
q1 = pd.read_sql_query('''
    SELECT symbol, COUNT(*) AS trading_days
    FROM price_data
    GROUP BY symbol
    ORDER BY trading_days DESC
''', conn)
print('Q1 — Trading days per ticker:')
print(q1.to_string(index=False))

In [ ]:
# Q2: Latest close price per ticker (correlated subquery on MAX(date))
q2 = pd.read_sql_query('''
    SELECT p.symbol, p.date, ROUND(p.close, 2) AS close
    FROM price_data p
    WHERE p.date = (SELECT MAX(date) FROM price_data WHERE symbol = p.symbol)
    ORDER BY p.symbol
''', conn)
print('Q2 — Latest close per ticker:')
print(q2.to_string(index=False))

In [ ]:
# Q3: Average GDELT sentiment + total article volume per ticker
q3 = pd.read_sql_query('''
    SELECT symbol,
           ROUND(AVG(sentiment_compound), 4) AS mean_sentiment,
           ROUND(MIN(sentiment_compound), 4) AS min_sentiment,
           ROUND(MAX(sentiment_compound), 4) AS max_sentiment,
           SUM(article_count) AS total_articles
    FROM news_articles
    GROUP BY symbol
    ORDER BY symbol
''', conn)
print('Q3 — Average GDELT sentiment per ticker:')
print(q3.to_string(index=False))

In [ ]:
# Q4: Anomaly count by classifier label
q4 = pd.read_sql_query('''
    SELECT label, COUNT(*) AS n
    FROM anomalies
    WHERE agreement_score >= 2
    GROUP BY label
    ORDER BY n DESC
''', conn)
total = q4['n'].sum()
q4['pct'] = (q4['n'] / total * 100).round(1)
print(f'Q4 — Anomaly count by classifier label (total = {total}):')
print(q4.to_string(index=False))

In [ ]:
# Q5: High-confidence anomalies per ticker (with consensus subset)
q5 = pd.read_sql_query('''
    SELECT symbol,
           COUNT(*) AS total_anomalies,
           ROUND(AVG(confidence), 3) AS avg_confidence,
           SUM(CASE WHEN agreement_score = 3 THEN 1 ELSE 0 END) AS consensus_anomalies
    FROM anomalies
    WHERE agreement_score >= 2
    GROUP BY symbol
    ORDER BY total_anomalies DESC
''', conn)
print('Q5 — Anomalies per ticker:')
print(q5.to_string(index=False))

In [ ]:
# Q6: Date coverage per ticker
q6 = pd.read_sql_query('''
    SELECT symbol,
           MIN(date) AS first_date,
           MAX(date) AS last_date,
           COUNT(*) AS days
    FROM price_data
    GROUP BY symbol
    ORDER BY symbol
''', conn)
print('Q6 — Date coverage per ticker:')
print(q6.to_string(index=False))

## 2. Advanced SQL Analysis (5)

Five queries that go beyond basic aggregation: window-style ordering, multi-table joins, date functions, cross-tab grouping, and self-joins.

In [ ]:
# Q7: Top-10 highest-agreement anomalies, ordered by agreement then confidence
q7 = pd.read_sql_query('''
    SELECT symbol, anomaly_date, agreement_score, ROUND(confidence,3) AS confidence,
           label, ROUND(price_change_1d,4) AS pc_1d, ROUND(price_change_5d,4) AS pc_5d
    FROM anomalies
    WHERE agreement_score >= 2
    ORDER BY agreement_score DESC, confidence DESC
    LIMIT 10
''', conn)
print('Q7 — Top 10 highest-agreement anomalies:')
print(q7.to_string(index=False))

In [ ]:
# Q8: Three-way JOIN — anomalies, price_data, news_articles (sentiment-spike subset)
q8 = pd.read_sql_query('''
    SELECT a.symbol, a.anomaly_date, a.label, ROUND(a.confidence,3) AS confidence,
           ROUND(p.close,2) AS close_price,
           ROUND(n.sentiment_compound,4) AS news_sentiment,
           n.article_count
    FROM anomalies a
    LEFT JOIN price_data p
      ON a.symbol = p.symbol AND a.anomaly_date = p.date
    LEFT JOIN news_articles n
      ON a.symbol = n.symbol AND DATE(a.anomaly_date) = DATE(n.published_at)
    WHERE a.agreement_score >= 2 AND a.label = 'vader_sentiment_spike'
    ORDER BY a.confidence DESC
''', conn)
print('Q8 — Sentiment-spike anomalies joined with price + news:')
print(q8.to_string(index=False))

In [ ]:
# Q9: Per-month anomaly density (date function + GROUP BY)
q9 = pd.read_sql_query('''
    SELECT strftime('%Y-%m', anomaly_date) AS month,
           COUNT(*) AS n_anomalies
    FROM anomalies
    WHERE agreement_score >= 2
    GROUP BY month
    ORDER BY n_anomalies DESC
    LIMIT 15
''', conn)
print('Q9 — Top-15 highest-anomaly months:')
print(q9.to_string(index=False))

In [ ]:
# Q10: Cross-tabulation — classifier label x ticker (GROUP BY two cols, then pivot)
q10 = pd.read_sql_query('''
    SELECT symbol, label, COUNT(*) AS n
    FROM anomalies
    WHERE agreement_score >= 2
    GROUP BY symbol, label
    ORDER BY symbol, n DESC
''', conn)
q10_pivot = q10.pivot(index='symbol', columns='label', values='n').fillna(0).astype(int)
print('Q10 — Classifier label distribution per ticker:')
print(q10_pivot)

In [ ]:
# Q11: Self-join — dates where 3+ tickers had simultaneous anomalies (contagion)
q11 = pd.read_sql_query('''
    SELECT anomaly_date,
           COUNT(DISTINCT symbol) AS n_tickers,
           GROUP_CONCAT(symbol) AS tickers
    FROM anomalies
    WHERE agreement_score >= 2
    GROUP BY anomaly_date
    HAVING COUNT(DISTINCT symbol) >= 3
    ORDER BY n_tickers DESC, anomaly_date
    LIMIT 20
''', conn)
print('Q11 — Top 20 contagion dates (3+ tickers anomalous on same day):')
print(q11.to_string(index=False))

## 3. Anomaly Distribution Visualizations (3)

Visualizations 6–8 from the report's Section 5: how anomalies are distributed across tickers, classifier labels, and time.

In [ ]:
# Vis 6: Anomaly count per ticker
counts = q5.set_index('symbol')['total_anomalies']
plt.figure(figsize=(10, 5))
bars = plt.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
for bar, v in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, v + 5, str(int(v)),
             ha='center', fontsize=10)
plt.title('High-Confidence Anomalies per Ticker (2015–2024)')
plt.ylabel('Anomaly count (agreement ≥ 2)')
plt.xlabel('Ticker')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Vis 7: Stacked bar of classifier label distribution per ticker
label_order = ['sector_contagion', 'macroeconomic_event', 'vader_sentiment_spike', 'unexplained']
ordered = q10_pivot.reindex(columns=[c for c in label_order if c in q10_pivot.columns])
ax = ordered.plot(kind='bar', stacked=True, figsize=(10, 5),
                  colormap='Set2', edgecolor='white')
ax.set_title('Anomaly Classification by Ticker')
ax.set_ylabel('Anomaly count')
ax.set_xlabel('Ticker')
plt.xticks(rotation=0)
plt.legend(title='Label', loc='upper right', fontsize=9)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Vis 8: Anomaly density timeline (full series, monthly)
density = pd.read_sql_query('''
    SELECT strftime('%Y-%m', anomaly_date) AS month, COUNT(*) AS n
    FROM anomalies WHERE agreement_score >= 2
    GROUP BY month ORDER BY month
''', conn)
density['month'] = pd.to_datetime(density['month'])
plt.figure(figsize=(14, 5))
plt.plot(density['month'], density['n'], marker='o', markersize=3, color='steelblue')
plt.fill_between(density['month'], 0, density['n'], alpha=0.3)
plt.title('Monthly Anomaly Density Across All Tickers (2015–2024)')
plt.ylabel('Anomalies per month')
plt.xlabel('Date')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
peak = density.loc[density['n'].idxmax()]
print(f'Peak month: {peak["month"].strftime("%Y-%m")} with {int(peak["n"])} anomalies')

## 4. Context Alignment Visualizations (3)

Visualizations 9–11 from the report's Section 5: alignment of anomalies with FOMC meetings, contagion events, and cross-ticker return correlation.

In [ ]:
# Vis 9: FOMC alignment — anomalies in ±2-day window of each FOMC meeting
from src.fomc_events import get_fomc_dates
fomc_list = get_fomc_dates()
fomc_df = pd.DataFrame({'fomc_date': pd.to_datetime(fomc_list)})
anom_df = pd.read_sql_query(
    'SELECT symbol, anomaly_date FROM anomalies WHERE agreement_score >= 2', conn)
anom_df['anomaly_date'] = pd.to_datetime(anom_df['anomaly_date'])

fomc_counts = []
for fd in fomc_df['fomc_date']:
    mask = (anom_df['anomaly_date'] >= fd - pd.Timedelta(days=2)) & \
           (anom_df['anomaly_date'] <= fd + pd.Timedelta(days=2))
    fomc_counts.append({'fomc_date': fd, 'anomaly_count': int(mask.sum())})
fomc_counts = pd.DataFrame(fomc_counts)

plt.figure(figsize=(14, 5))
plt.bar(fomc_counts['fomc_date'], fomc_counts['anomaly_count'],
        width=20, color='#e74c3c', alpha=0.7)
plt.title('Anomalies in ±2-Day Window of Each FOMC Meeting (2015–2024)')
plt.ylabel('Anomaly count')
plt.xlabel('FOMC meeting date')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print(f'FOMC meetings: {len(fomc_counts)}')
print(f'Mean anomalies per FOMC ±2-day window: {fomc_counts["anomaly_count"].mean():.2f}')
print(f'Max anomalies in a single window: {fomc_counts["anomaly_count"].max()}')

In [ ]:
# Vis 10: Contagion timeline — days where >= 3 tickers were simultaneously anomalous
contagion_full = pd.read_sql_query('''
    SELECT anomaly_date, COUNT(DISTINCT symbol) AS n_tickers
    FROM anomalies
    WHERE agreement_score >= 2
    GROUP BY anomaly_date
    HAVING COUNT(DISTINCT symbol) >= 3
    ORDER BY anomaly_date
''', conn)
contagion_full['anomaly_date'] = pd.to_datetime(contagion_full['anomaly_date'])

plt.figure(figsize=(14, 5))
colors = {3: '#3498db', 4: '#f39c12', 5: '#e74c3c'}
for n in [3, 4, 5]:
    sub = contagion_full[contagion_full['n_tickers'] == n]
    if len(sub) > 0:
        plt.scatter(sub['anomaly_date'], [n] * len(sub),
                    c=colors[n], s=30, label=f'{n} tickers', alpha=0.7)
plt.title('Sector Contagion Timeline: Days with ≥3 Tickers Anomalous')
plt.ylabel('Number of tickers with anomaly')
plt.xlabel('Date')
plt.yticks([3, 4, 5])
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Total contagion dates: {len(contagion_full)}')
print(f'Days with all 5 tickers anomalous: {int((contagion_full["n_tickers"]==5).sum())}')

In [ ]:
# Vis 11: Cross-ticker daily-return correlation heatmap
# Verifies the correlation values cited in the report's Section 5.
prices = pd.read_sql_query('SELECT symbol, date, close FROM price_data', conn)
prices['date'] = pd.to_datetime(prices['date'], utc=True).dt.tz_localize(None)
pivot = prices.pivot(index='date', columns='symbol', values='close')
returns = pivot.pct_change().dropna()

corr = returns.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={'label': 'Pearson correlation'})
plt.title('Daily-Return Correlation Across Tickers (2015–2024)')
plt.tight_layout()
plt.show()
print('Correlation matrix:')
print(corr.round(3))
print(f'\nTrading days in correlation sample: {len(returns)}')
print('\nDaily-return std (volatility per ticker):')
print(returns.std().round(4))

## 5. Methodology and CS 210 Connections

### Relational Database Design
Four normalized SQLite tables: `stocks`, `price_data`, `news_articles`, `anomalies`. Foreign keys on `symbol`, UNIQUE constraints on `(symbol, date)` and `(symbol, anomaly_date)`, and indexes on the per-ticker date columns. The schema supports the multi-table joins shown above (Q8) without denormalization.

### Data Cleaning
Eight explicit data quality steps split across `data_ingestion.py` and `news_bq.py`: schema validation, GDELT origin clamping, unique-key enforcement, feature warm-up drop (rows where the 200-day SMA has not yet stabilized), column-name normalization, `Adj Close` synthesis when yfinance omits it, V2Tone scaling from [-100,100] to [-1,1], and timezone stripping for news-vs-anomaly date comparisons.

### Unsupervised ML
No labeled anomaly ground truth exists. Three complementary detectors are combined via agreement scoring:
- **Z-Score** — univariate statistical deviation; flags any feature beyond 3σ.
- **Isolation Forest** — random-tree partitioning; rare points get isolated in fewer splits.
- **LOF** — density-based; flags points with substantially lower local density than their neighbors.

An anomaly is *high-confidence* only when ≥2 of the three methods agree, which reduces false positives that any single noisy detector would produce alone. The classifier then assigns one of four semantic labels with strict precedence: `macroeconomic_event` → `sector_contagion` → `vader_sentiment_spike` → `unexplained`.

## 6. Limitations and Future Work

1. **GDELT 2.0 GKG starts Feb 18, 2015.** The first ~6 weeks of the analysis window have no news rows, so `vader_sentiment_spike` cannot fire there.
2. **V2Tone is corpus-aggregated.** Daily averages smooth individual outlier articles before they reach the classifier — a single highly-negative article on an otherwise neutral day will not register.
3. **No earnings calendar.** Most of the 27.9% `unexplained` anomalies are likely earnings-driven; future work should add a structured earnings-date list as a fifth label.
4. **Survivorship bias.** Only currently-listed tickers were analyzed; delisted/bankrupt names (which would presumably contain the most extreme anomalies) are absent.
5. **Contamination hyperparameter.** Both Isolation Forest and LOF assume `contamination=0.1`; a different value would directly change the size of the anomaly set. No formal tuning was performed.
6. **Per-ticker z-score baseline uses full history.** The sentiment-spike test compares each window against statistics computed from the entire series, including future data. A walk-forward baseline would be more rigorous.
7. **Five tickers is a small universe.** Sector-level patterns (tech vs energy vs financials) cannot be tested with the current ticker set, which is tech-heavy.

**Future work:** add an earnings-calendar label, switch to per-article (not daily) V2Tone, expand to a sector-diverse 50+ ticker universe, add a GARCH-family volatility detector, and implement walk-forward sentiment baselining.

In [ ]:
conn.close()
print('Notebook complete.')